# Whisper → ONNX → TensorRT (whisper2)

`whisper.ipynb`를 정리·수정한 버전. 위에서 아래로 한 번에 돌아가도록 구성.

원본 대비 수정 사항
- **TRT 인코더 입력 버그**: CPU 텐서를 넣으면 `WhisperTRTEncoder.forward`가 mel 특징을 `.int()`로 캐스팅 → 인코더 출력이 틀어짐 ("Encoder output이 다름!!"의 원인). `Whisper/trt.py`에서 `.float()`로 수정.
- **TRT e2e 결과 불일치**: 원본은 TRT 디코더에 *한국어 뉴스*의 인코더 출력(`encoder_outputs`)을 넘기고 librispeech 정답과 비교해서 `False` / `" you"`가 나왔음. 같은 오디오의 TRT 인코더 출력을 쓰도록 수정.
- `cuda:1`/`cuda` 혼용 제거: `CUDA_VISIBLE_DEVICES`로 GPU 하나만 보이게 하고 전부 `cuda`로 통일.
- NNDF가 `archived_models/`로 옮겨져서 import 경로 추가.
- ONNX는 FP32로 한 번만 export하고 FP32/FP16 엔진 둘 다 같은 ONNX에서 빌드.

환경: conda `trt` (python 3.8, torch 2.0.1+cu117, transformers 4.24.0, tensorrt 8.6.1, polygraphy 0.48.1, onnx 1.14.1)

In [1]:
import os, sys

# 사용할 GPU 하나만 노출 (torch / tensorrt import 전에 설정해야 함)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("CUDA_MODULE_LOADING", "LAZY")

ROOT_DIR = os.path.abspath("../")
for p in (ROOT_DIR, os.path.join(ROOT_DIR, "archived_models")):
    if p not in sys.path:
        sys.path.append(p)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import tensorrt as trt
import transformers

from transformers import WhisperProcessor, WhisperForConditionalGeneration, WhisperConfig

DEVICE = "cuda"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| tensorrt", trt.__version__)
print("GPU:", torch.cuda.get_device_name(0))

torch 2.0.1+cu117 | transformers 4.24.0 | tensorrt 8.6.1
GPU: NVIDIA A100-SXM4-80GB


## 1. HuggingFace Whisper 모델 로드

In [2]:
# choices: openai/whisper-tiny | base | small | medium | large-v2
Whisper_VARIANT = os.environ.get("WHISPER_VARIANT", "openai/whisper-large-v2")

processor = WhisperProcessor.from_pretrained(Whisper_VARIANT)
tokenizer = processor.tokenizer
whisper_model = WhisperForConditionalGeneration.from_pretrained(Whisper_VARIANT).eval()
config = whisper_model.config

pytorch_model_dir = f"./models/{Whisper_VARIANT}/pytorch"
os.makedirs(pytorch_model_dir, exist_ok=True)
whisper_model.save_pretrained(pytorch_model_dir)
print("Pytorch Model saved to", pytorch_model_dir)

preprocessor_config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 185kB [00:00, 49.8MB/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 283kB [00:00, 249MB/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin: 100%|█████████▉| 6.16G/6.17G [02:27<00:00, 43.1MB/s]

pytorch_model.bin: 100%|█████████▉| 6.17G/6.17G [02:27<00:00, 43.1MB/s]

pytorch_model.bin: 100%|██████████| 6.17G/6.17G [02:27<00:00, 42.8MB/s]

pytorch_model.bin: 100%|██████████| 6.17G/6.17G [02:27<00:00, 41.7MB/s]

Pytorch Model saved to ./models/openai/whisper-large-v2/pytorch


## 2. 입력 오디오

- 영어: librispeech dummy 샘플
- 한국어: `korean_news.mp4` (PyAV로 16kHz mono 디코딩)

In [3]:
import io, itertools
import av

def decode_audio(input_file, sampling_rate=16000):
    """PyAV로 오디오/비디오 파일을 16kHz mono float32로 디코딩."""
    resampler = av.audio.resampler.AudioResampler(format="s16", layout="mono", rate=sampling_rate)
    raw_buffer = io.BytesIO()
    dtype = None
    with av.open(input_file, metadata_errors="ignore") as container:
        frames = container.decode(audio=0)
        frames = _ignore_invalid_frames(frames)
        frames = _group_frames(frames, 500000)
        frames = _resample_frames(frames, resampler)
        for frame in frames:
            array = frame.to_ndarray()
            dtype = array.dtype
            raw_buffer.write(array)
    audio = np.frombuffer(raw_buffer.getbuffer(), dtype=dtype)
    return audio.astype(np.float32) / 32768.0

def _ignore_invalid_frames(frames):
    iterator = iter(frames)
    while True:
        try:
            yield next(iterator)
        except StopIteration:
            break
        except av.error.InvalidDataError:
            continue

def _group_frames(frames, num_samples=None):
    fifo = av.audio.fifo.AudioFifo()
    for frame in frames:
        frame.pts = None  # Ignore timestamp check.
        fifo.write(frame)
        if num_samples is not None and fifo.samples >= num_samples:
            yield fifo.read()
    if fifo.samples > 0:
        yield fifo.read()

def _resample_frames(frames, resampler):
    # Add None to flush the resampler.
    for frame in itertools.chain(frames, [None]):
        yield from resampler.resample(frame)

In [4]:
from datasets import load_dataset

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
input_features = processor(ds[0]["audio"]["array"], sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)

ko_audio = decode_audio("korean_news.mp4")
print(f"korean_news.mp4: {ko_audio.shape[0] / 16000:.1f}s")
input_features_ko = processor(ko_audio, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)

forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="transcribe", no_timestamps=True)
forced_decoder_ids_ko = processor.get_decoder_prompt_ids(language="ko", task="transcribe", no_timestamps=True)
input_features.shape, input_features_ko.shape

korean_news.mp4: 81.2s


(torch.Size([1, 80, 3000]), torch.Size([1, 80, 3000]))

## 3. PyTorch(HF) 기준 결과

In [5]:
whisper_model.to(DEVICE).float()
max_output_len = config.max_length
min_output_len = 0
num_beams = 1

def hf_generate(features, forced_ids, use_cache=True):
    return whisper_model.generate(
        features, forced_decoder_ids=forced_ids, max_length=max_output_len,
        min_length=min_output_len, num_beams=num_beams, use_cache=use_cache,
    )

with torch.no_grad():
    outputs_hf = tokenizer.decode(hf_generate(input_features, forced_decoder_ids)[0], skip_special_tokens=True)
    outputs_hf_ko = tokenizer.decode(hf_generate(input_features_ko, forced_decoder_ids_ko)[0], skip_special_tokens=True)
print("EN:", outputs_hf)
print("KO:", outputs_hf_ko)

EN:  Mr. Quilter is the apostle of the middle classes and we are glad to welcome his gospel.
KO:  제6호 태풍 카누는 여전히 매우 강한 세력을 유지한 채 북서지나고 있습니다. 하지만 이동 속도가 점점 느려져 거의 정체하는 모습입니다. 태풍은 동중국해에 머물다 동쪽으로 방향을 급격히 틀어 이동할 걸로 보입니다. 속도도 조금씩 빨라지며 다음 주 초중반에는 일본 규슈 남쪽 해상까지 진출하겠습니다. 세력도 크게 약화하지는 않을 전망입니다.


### HF generate 벤치마크

In [6]:
from NNDF.networks import TimingProfile, NetworkMetadata, Precision
from NNDF.general_utils import measure_python_inference_code

timing_profile = TimingProfile(iterations=10, number=1, warmup=1, duration=0, percentile=[50, 99])

def percentile_print(timing):
    return ", ".join(f"p{timing_profile.percentile[i]} {p * 1000:.2f}ms" for i, p in enumerate(timing))

with torch.no_grad():
    whisper_model.float()
    hf_nonkv_time = measure_python_inference_code(lambda: hf_generate(input_features, forced_decoder_ids, use_cache=False), timing_profile)
    hf_kv_time = measure_python_inference_code(lambda: hf_generate(input_features, forced_decoder_ids, use_cache=True), timing_profile)

    whisper_model.half()
    hf_nonkv_time_fp16 = measure_python_inference_code(lambda: hf_generate(input_features.half(), forced_decoder_ids, use_cache=False), timing_profile)
    hf_kv_time_fp16 = measure_python_inference_code(lambda: hf_generate(input_features.half(), forced_decoder_ids, use_cache=True), timing_profile)
    whisper_model.float()

print("HF FP32 no-cache:", percentile_print(hf_nonkv_time), "| cache:", percentile_print(hf_kv_time))
print("HF FP16 no-cache:", percentile_print(hf_nonkv_time_fp16), "| cache:", percentile_print(hf_kv_time_fp16))

HF FP32 no-cache: p50 1121.82ms, p99 1134.61ms | cache: p50 693.10ms, p99 704.27ms
HF FP16 no-cache: p50 775.88ms, p99 783.67ms | cache: p50 560.00ms, p99 565.43ms


### NNDF PyTorch 래퍼(인코더/디코더 분리) + greedy search

In [7]:
from Whisper.measurements import (
    decoder_inference as w_decoder_inference,
    encoder_inference as w_encoder_inference,
    full_inference_greedy,
)
from Whisper.export import WhisperEncoderTorchFile, WhisperDecoderTorchFile, WhisperEncoderTRTEngine, WhisperDecoderTRTEngine
from Whisper.WhisperModelConfig import WhisperModelTRTConfig, WhisperMetadata

HF_KV = True
decoder_input_ids = torch.full((1, 1), config.decoder_start_token_id, dtype=torch.int32, device=DEVICE)

def run_torch(precision_half):
    whisper_model.half() if precision_half else whisper_model.float()
    feats = input_features.half() if precision_half else input_features.float()
    enc = WhisperEncoderTorchFile.TorchModule(whisper_model.get_encoder())
    dec = WhisperDecoderTorchFile.TorchModule(whisper_model.get_decoder(), whisper_model.proj_out, config)
    with torch.no_grad():
        enc_out, enc_time = w_encoder_inference(enc, feats, timing_profile)
        _, dec_time = w_decoder_inference(dec, decoder_input_ids, enc_out, timing_profile, use_cache=HF_KV)
        ids, full_time = full_inference_greedy(
            enc, dec, feats, tokenizer, timing_profile,
            max_length=max_output_len, min_length=min_output_len,
            use_cache=HF_KV, forced_decoder_ids=forced_decoder_ids,
        )
    return tokenizer.decode(ids[0], skip_special_tokens=True), enc_time, dec_time, full_time

outputs_pytorch, encoder_pytorch_time, decoder_pytorch_time, full_pytorch_time = run_torch(False)
outputs_pytorch_fp16, encoder_pytorch_time_fp16, decoder_pytorch_time_fp16, full_pytorch_time_fp16 = run_torch(True)
whisper_model.float()

print(f"PyTorch FP32 == HF? {outputs_pytorch == outputs_hf}")
print(f"PyTorch FP16 == HF? {outputs_pytorch_fp16 == outputs_hf}")
print(f"FP32 enc {percentile_print(encoder_pytorch_time)} | dec {percentile_print(decoder_pytorch_time)} | e2e {percentile_print(full_pytorch_time)}")
print(f"FP16 enc {percentile_print(encoder_pytorch_time_fp16)} | dec {percentile_print(decoder_pytorch_time_fp16)} | e2e {percentile_print(full_pytorch_time_fp16)}")

PyTorch FP32 == HF? True
PyTorch FP16 == HF? True
FP32 enc p50 175.16ms, p99 176.05ms | dec p50 26.43ms, p99 52.76ms | e2e p50 706.55ms, p99 711.51ms
FP16 enc p50 45.64ms, p99 46.02ms | dec p50 24.16ms, p99 24.88ms | e2e p50 577.97ms, p99 582.56ms


## 4. ONNX 변환

인코더와 디코더(+LM head)를 따로 export. FP32로 한 번만 export하고 FP16 엔진도 같은 ONNX에서 빌드한다. (non-KV-cache 디코더)

In [8]:
TRT_KV = False
wh_onnx_model_path = f"./models/{Whisper_VARIANT}/onnx"
os.makedirs(wh_onnx_model_path, exist_ok=True)

trt_model_config = WhisperModelTRTConfig()
metadata = NetworkMetadata(variant=Whisper_VARIANT, precision=Precision(fp16=False), other=WhisperMetadata(kv_cache=TRT_KV))
metadata_fp16 = NetworkMetadata(variant=Whisper_VARIANT, precision=Precision(fp16=True), other=WhisperMetadata(kv_cache=TRT_KV))
metadata_string = trt_model_config.get_metadata_string(metadata)
metadata_string_fp16 = trt_model_config.get_metadata_string(metadata_fp16)

wh_encoder_onnx_fpath = os.path.join(wh_onnx_model_path, metadata_string + "-encoder.onnx")
wh_decoder_onnx_fpath = os.path.join(wh_onnx_model_path, metadata_string + "-decoder-with-lm-head.onnx")

force_onnx = False
if force_onnx or not (os.path.exists(wh_encoder_onnx_fpath) and os.path.exists(wh_decoder_onnx_fpath)):
    # ONNX export는 CPU + FP32에서
    whisper_model.float().cpu()
    WhisperEncoderTorchFile(whisper_model, metadata).as_onnx_model(wh_encoder_onnx_fpath, force_overwrite=True)
    WhisperDecoderTorchFile(whisper_model, metadata).as_onnx_model(wh_decoder_onnx_fpath, force_overwrite=True)
    whisper_model.to(DEVICE)
print(wh_encoder_onnx_fpath)
print(wh_decoder_onnx_fpath)

============= Diagnostic Run torch.onnx.export version 2.0.1+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



============= Diagnostic Run torch.onnx.export version 2.0.1+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



./models/openai/whisper-large-v2/onnx/Whisper-large-v2-encoder.onnx
./models/openai/whisper-large-v2/onnx/Whisper-large-v2-decoder-with-lm-head.onnx


## 5. TensorRT 엔진 빌드

동적 shape 최적화 프로파일 지정. 엔진 파일이 있으면 재사용 (`force_build=True`로 재빌드).

In [9]:
from Whisper.export import WhisperDecoderONNXFile, WhisperEncoderONNXFile
from polygraphy.backend.trt import Profile
from tensorrt import PreviewFeature

batch_size = 1
encoder_hidden_size = config.d_model
max_source_positions = config.max_source_positions  # 1500
max_sequence_length = WhisperModelTRTConfig.MAX_SEQUENCE_LENGTH[Whisper_VARIANT]

decoder_profile = Profile()
decoder_profile.add(
    "input_ids",
    min=(batch_size * num_beams, 1),
    opt=(batch_size * num_beams, max_sequence_length // 2),
    max=(batch_size * num_beams, max_sequence_length),
)
decoder_profile.add(
    "encoder_hidden_states",
    min=(batch_size * num_beams, 1, encoder_hidden_size),
    opt=(batch_size * num_beams, max_source_positions, encoder_hidden_size),
    max=(batch_size * num_beams, max_source_positions, encoder_hidden_size),
)
encoder_profile = Profile()
encoder_profile.add(
    "input_features",
    min=(batch_size, config.num_mel_bins, 3000),
    opt=(batch_size, config.num_mel_bins, 3000),
    max=(batch_size, config.num_mel_bins, 3000),
)

preview_features = [PreviewFeature.DISABLE_EXTERNAL_TACTIC_SOURCES_FOR_CORE_0805, PreviewFeature.FASTER_DYNAMIC_SHAPES_0805]
engine_tag = f"bs{batch_size}" + (f"-beam{num_beams}" if num_beams > 1 else "")

wh_tensorrt_model_path = f"./models/{Whisper_VARIANT}/tensorrt"
os.makedirs(wh_tensorrt_model_path, exist_ok=True)
force_build = False

def build_engines(meta, meta_string):
    enc_path = os.path.join(wh_tensorrt_model_path, f"{meta_string}-encoder.onnx-bs{batch_size}.engine")
    dec_path = os.path.join(wh_tensorrt_model_path, f"{meta_string}-decoder-with-lm-head.onnx-{engine_tag}.engine")
    if force_build or not os.path.exists(enc_path):
        enc = WhisperEncoderONNXFile(wh_encoder_onnx_fpath, meta).as_trt_engine(
            enc_path, force_overwrite=True, profiles=[encoder_profile], preview_features=preview_features)
    else:
        enc = WhisperEncoderTRTEngine(enc_path, meta)
    if force_build or not os.path.exists(dec_path):
        dec = WhisperDecoderONNXFile(wh_decoder_onnx_fpath, meta).as_trt_engine(
            dec_path, force_overwrite=True, profiles=[decoder_profile], preview_features=preview_features)
    else:
        dec = WhisperDecoderTRTEngine(dec_path, meta)
    print(enc_path, "\n", dec_path)
    return enc, dec

whisper_trt_encoder_engine, whisper_trt_decoder_engine = build_engines(metadata, metadata_string)
whisper_trt_encoder_engine_fp16, whisper_trt_decoder_engine_fp16 = build_engines(metadata_fp16, metadata_string_fp16)

[W] 'colored' module is not installed, will not use colors when logging. To enable colors, please install the 'colored' module: python3 -m pip install colored
[W] onnx2trt_utils.cpp:374: Your ONNX model has been generated with INT64 weights, while TensorRT does not natively support INT64. Attempting to cast down to INT32.


[W] It looks like some layers in the network have compute precision set, but precision constraints were not enabled. 
    Precision constraints must be set to 'prefer' or 'obey' for layer compute precision to take effect. 
    Note: Layers and their requested precisions were: {'encoder/layers.0/self_attn_layer_norm/ReduceMean': 'FLOAT', 'encoder/layers.0/self_attn_layer_norm/Pow': 'FLOAT', 'encoder/layers.0/self_attn_layer_norm/ReduceMean_1': 'FLOAT', 'encoder/layers.0/self_attn_layer_norm/Add': 'FLOAT', 'encoder/layers.0/self_attn_layer_norm/Sqrt': 'FLOAT', 'encoder/layers.0/self_attn_layer_norm/Div': 'FLOAT', 'encoder/layers.0/self_attn_layer_norm/Mul': 'FLOAT', 'encoder/layers.0/final_layer_norm/ReduceMean': 'FLOAT', 'encoder/layers.0/final_layer_norm/Pow': 'FLOAT', 'encoder/layers.0/final_layer_norm/ReduceMean_1': 'FLOAT', 'encoder/layers.0/final_layer_norm/Add': 'FLOAT', 'encoder/layers.0/final_layer_norm/Sqrt': 'FLOAT', 'encoder/layers.0/final_layer_norm/Div': 'FLOAT', 'encoder/l

[W] onnx2trt_utils.cpp:400: One or more weights outside the range of INT32 was clamped


[W] It looks like some layers in the network have compute precision set, but precision constraints were not enabled. 
    Precision constraints must be set to 'prefer' or 'obey' for layer compute precision to take effect. 
    Note: Layers and their requested precisions were: {'/decoder/layers.0/self_attn_layer_norm/ReduceMean': 'FLOAT', '/decoder/layers.0/self_attn_layer_norm/Pow': 'FLOAT', '/decoder/layers.0/self_attn_layer_norm/ReduceMean_1': 'FLOAT', '/decoder/layers.0/self_attn_layer_norm/Add': 'FLOAT', '/decoder/layers.0/self_attn_layer_norm/Sqrt': 'FLOAT', '/decoder/layers.0/self_attn_layer_norm/Div': 'FLOAT', '/decoder/layers.0/self_attn_layer_norm/Mul': 'FLOAT', '/decoder/layers.0/encoder_attn_layer_norm/ReduceMean': 'FLOAT', '/decoder/layers.0/encoder_attn_layer_norm/Pow': 'FLOAT', '/decoder/layers.0/encoder_attn_layer_norm/ReduceMean_1': 'FLOAT', '/decoder/layers.0/encoder_attn_layer_norm/Add': 'FLOAT', '/decoder/layers.0/encoder_attn_layer_norm/Sqrt': 'FLOAT', '/decoder/lay

./models/openai/whisper-large-v2/tensorrt/Whisper-large-v2-encoder.onnx-bs1.engine 
 ./models/openai/whisper-large-v2/tensorrt/Whisper-large-v2-decoder-with-lm-head.onnx-bs1.engine


[W] It looks like some layers in the network have compute precision set, but precision constraints were not enabled. 
    Precision constraints must be set to 'prefer' or 'obey' for layer compute precision to take effect. 
    Note: Layers and their requested precisions were: {'encoder/layers.0/self_attn_layer_norm/ReduceMean': 'FLOAT', 'encoder/layers.0/self_attn_layer_norm/Pow': 'FLOAT', 'encoder/layers.0/self_attn_layer_norm/ReduceMean_1': 'FLOAT', 'encoder/layers.0/self_attn_layer_norm/Add': 'FLOAT', 'encoder/layers.0/self_attn_layer_norm/Sqrt': 'FLOAT', 'encoder/layers.0/self_attn_layer_norm/Div': 'FLOAT', 'encoder/layers.0/self_attn_layer_norm/Mul': 'FLOAT', 'encoder/layers.0/final_layer_norm/ReduceMean': 'FLOAT', 'encoder/layers.0/final_layer_norm/Pow': 'FLOAT', 'encoder/layers.0/final_layer_norm/ReduceMean_1': 'FLOAT', 'encoder/layers.0/final_layer_norm/Add': 'FLOAT', 'encoder/layers.0/final_layer_norm/Sqrt': 'FLOAT', 'encoder/layers.0/final_layer_norm/Div': 'FLOAT', 'encoder/l

[W] Detected layernorm nodes in FP16: , , , , /decoder/layers.0/self_attn_layer_norm/Sub, , , , , , /decoder/layers.0/self_attn_layer_norm/Add_1, /decoder/layers.0/encoder_attn_layer_norm/Sub, , , , , , /decoder/layers.0/encoder_attn_layer_norm/Add_1, /decoder/layers.0/final_layer_norm/Sub, , , , , , /decoder/layers.0/final_layer_norm/Add_1, /decoder/layers.1/self_attn_layer_norm/Sub, , , , , , /decoder/layers.1/self_attn_layer_norm/Add_1, /decoder/layers.1/encoder_attn_layer_norm/Sub, , , , , , /decoder/layers.1/encoder_attn_layer_norm/Add_1, /decoder/layers.1/final_layer_norm/Sub, , , , , , /decoder/layers.1/final_layer_norm/Add_1, /decoder/layers.2/self_attn_layer_norm/Sub, , , , , , /decoder/layers.2/self_attn_layer_norm/Add_1, /decoder/layers.2/encoder_attn_layer_norm/Sub, , , , , , /decoder/layers.2/encoder_attn_layer_norm/Add_1, /decoder/layers.2/final_layer_norm/Sub, , , , , , /decoder/layers.2/final_layer_norm/Add_1, /decoder/layers.3/self_attn_layer_norm/Sub, , , , , , /decod

[W] TensorRT encountered issues when converting weights between types and that could affect accuracy.
[W] If this is not the desired behavior, please modify the weights or retrain with regularization to adjust the magnitude of the weights.
[W] Check verbose logs for the list of affected weights.
[W] - 727 weights are affected by this issue: Detected subnormal FP16 values.
[W] - 1 weights are affected by this issue: Detected finite FP32 values which would overflow in FP16 and converted them to the closest finite FP16 value.


./models/openai/whisper-large-v2/tensorrt/Whisper-large-v2-fp16-encoder.onnx-bs1.engine 
 ./models/openai/whisper-large-v2/tensorrt/Whisper-large-v2-fp16-decoder-with-lm-head.onnx-bs1.engine


## 6. TensorRT 추론

In [10]:
from Whisper.trt import WhisperTRTEncoder, WhisperTRTDecoder

trt_config = WhisperConfig.from_pretrained(Whisper_VARIANT, use_cache=metadata.other.kv_cache)

whisper_trt_encoder = WhisperTRTEncoder(whisper_trt_encoder_engine, metadata, trt_config, batch_size=batch_size)
whisper_trt_decoder = WhisperTRTDecoder(whisper_trt_decoder_engine, metadata, trt_config, batch_size=batch_size, num_beams=num_beams)
whisper_trt_encoder_fp16 = WhisperTRTEncoder(whisper_trt_encoder_engine_fp16, metadata_fp16, trt_config, batch_size=batch_size)
whisper_trt_decoder_fp16 = WhisperTRTDecoder(whisper_trt_decoder_engine_fp16, metadata_fp16, trt_config, batch_size=batch_size, num_beams=num_beams)

### 인코더 출력 비교 (PyTorch vs TRT)

원본에서 "Encoder output이 다름!!"이던 부분. GPU 입력/CPU 입력 둘 다 PyTorch와 일치해야 한다.

In [11]:
with torch.no_grad():
    ref = whisper_model.get_encoder()(input_features_ko.float()).last_hidden_state
    for name, enc in [("FP32", whisper_trt_encoder), ("FP16", whisper_trt_encoder_fp16)]:
        out_gpu = enc(input_features=input_features_ko).clone()
        out_cpu = enc(input_features=input_features_ko.cpu()).clone()
        print(f"TRT {name} encoder | max|diff| gpu-in {(out_gpu - ref).abs().max().item():.4f}"
              f" | cpu-in {(out_cpu.to(DEVICE) - ref).abs().max().item():.4f}")

TRT FP32 encoder | max|diff| gpu-in 0.0957 | cpu-in 0.0957
TRT FP16 encoder | max|diff| gpu-in 0.0957 | cpu-in 0.0957


### End-to-end greedy decoding (영어 / 한국어)

In [12]:
from transformers.generation_logits_process import (
    LogitsProcessorList,
    SuppressTokensAtBeginLogitsProcessor,
    SuppressTokensLogitsProcessor,
    ForceTokensLogitsProcessor,
)
from transformers.generation_stopping_criteria import MaxLengthCriteria, StoppingCriteriaList

stopping_criteria = StoppingCriteriaList([MaxLengthCriteria(max_output_len)])

def make_logits_processor(forced_ids):
    return LogitsProcessorList([
        SuppressTokensLogitsProcessor(config.suppress_tokens),
        SuppressTokensAtBeginLogitsProcessor(config.begin_suppress_tokens, decoder_input_ids.shape[-1]),
        ForceTokensLogitsProcessor(forced_ids),
    ])

def trt_e2e(trt_encoder, trt_decoder, features, forced_ids):
    with torch.no_grad():
        # 반드시 같은 오디오의 인코더 출력을 디코더에 넘긴다 (원본 버그 지점)
        encoder_last_hidden_states = trt_encoder(input_features=features)
        trt_decoder.set_encoder_hidden_states_for_inference_cycle(encoder_last_hidden_states)
        return trt_decoder.greedy_search(
            input_ids=decoder_input_ids,
            encoder_hidden_states=encoder_last_hidden_states,
            stopping_criteria=stopping_criteria,
            logits_processor=make_logits_processor(forced_ids),
            use_cache=metadata.other.kv_cache,
        )

results = {}
for name, enc, dec in [("FP32", whisper_trt_encoder, whisper_trt_decoder), ("FP16", whisper_trt_encoder_fp16, whisper_trt_decoder_fp16)]:
    en = tokenizer.decode(trt_e2e(enc, dec, input_features, forced_decoder_ids)[0], skip_special_tokens=True)
    ko = tokenizer.decode(trt_e2e(enc, dec, input_features_ko, forced_decoder_ids_ko)[0], skip_special_tokens=True)
    results[name] = (en, ko)
    print(f"[TRT {name}] EN == HF? {en == outputs_hf} | {en}")
    print(f"[TRT {name}] KO == HF? {ko == outputs_hf_ko} | {ko}")

outputs_trt, outputs_trt_ko = results["FP32"]
outputs_trt_fp16, outputs_trt_fp16_ko = results["FP16"]

[TRT FP32] EN == HF? True |  Mr. Quilter is the apostle of the middle classes and we are glad to welcome his gospel.
[TRT FP32] KO == HF? True |  제6호 태풍 카누는 여전히 매우 강한 세력을 유지한 채 북서지나고 있습니다. 하지만 이동 속도가 점점 느려져 거의 정체하는 모습입니다. 태풍은 동중국해에 머물다 동쪽으로 방향을 급격히 틀어 이동할 걸로 보입니다. 속도도 조금씩 빨라지며 다음 주 초중반에는 일본 규슈 남쪽 해상까지 진출하겠습니다. 세력도 크게 약화하지는 않을 전망입니다.


[TRT FP16] EN == HF? True |  Mr. Quilter is the apostle of the middle classes and we are glad to welcome his gospel.
[TRT FP16] KO == HF? True |  제6호 태풍 카누는 여전히 매우 강한 세력을 유지한 채 북서지나고 있습니다. 하지만 이동 속도가 점점 느려져 거의 정체하는 모습입니다. 태풍은 동중국해에 머물다 동쪽으로 방향을 급격히 틀어 이동할 걸로 보입니다. 속도도 조금씩 빨라지며 다음 주 초중반에는 일본 규슈 남쪽 해상까지 진출하겠습니다. 세력도 크게 약화하지는 않을 전망입니다.


### TRT 인코더 / 디코더 / E2E 시간 측정

In [13]:
def bench_trt(trt_encoder, trt_decoder):
    enc_out, enc_time = w_encoder_inference(trt_encoder, input_features, timing_profile)
    _, dec_time = w_decoder_inference(trt_decoder, decoder_input_ids, enc_out, timing_profile)
    _, full_time = full_inference_greedy(
        trt_encoder, trt_decoder, input_features, tokenizer, timing_profile,
        max_length=max_output_len, min_length=0, batch_size=batch_size,
        use_cache=metadata.other.kv_cache, forced_decoder_ids=forced_decoder_ids,
    )
    return enc_time, dec_time, full_time

encoder_trt_time, decoder_trt_time, full_trt_time = bench_trt(whisper_trt_encoder, whisper_trt_decoder)
encoder_trt_time_fp16, decoder_trt_time_fp16, full_trt_time_fp16 = bench_trt(whisper_trt_encoder_fp16, whisper_trt_decoder_fp16)

print(f"TRT FP32 enc {percentile_print(encoder_trt_time)} | dec {percentile_print(decoder_trt_time)} | e2e {percentile_print(full_trt_time)}")
print(f"TRT FP16 enc {percentile_print(encoder_trt_time_fp16)} | dec {percentile_print(decoder_trt_time_fp16)} | e2e {percentile_print(full_trt_time_fp16)}")

TRT FP32 enc p50 41.84ms, p99 42.10ms | dec p50 12.93ms, p99 13.51ms | e2e p50 404.66ms, p99 423.31ms
TRT FP16 enc p50 41.99ms, p99 42.55ms | dec p50 7.42ms, p99 7.50ms | e2e p50 242.83ms, p99 266.53ms


## 7. 결과 요약

In [14]:
from tabulate import tabulate

ms = lambda t: f"{t[0] * 1000:.2f}"
data = [
    ["Framework", "Precision", "Encoder p50 (ms)", "Decoder p50 (ms)", "Full E2E p50 (ms)", "EN == HF", "KO == HF"],
    ["HuggingFace (w/o cache)", "FP32", "-", "-", ms(hf_nonkv_time), "-", "-"],
    ["HuggingFace (w/ cache)", "FP32", "-", "-", ms(hf_kv_time), "-", "-"],
    ["HuggingFace (w/o cache)", "FP16", "-", "-", ms(hf_nonkv_time_fp16), "-", "-"],
    ["HuggingFace (w/ cache)", "FP16", "-", "-", ms(hf_kv_time_fp16), "-", "-"],
    ["PyTorch", "FP32", ms(encoder_pytorch_time), ms(decoder_pytorch_time), ms(full_pytorch_time), outputs_pytorch == outputs_hf, "-"],
    ["PyTorch", "FP16", ms(encoder_pytorch_time_fp16), ms(decoder_pytorch_time_fp16), ms(full_pytorch_time_fp16), outputs_pytorch_fp16 == outputs_hf, "-"],
    ["TensorRT", "FP32", ms(encoder_trt_time), ms(decoder_trt_time), ms(full_trt_time), outputs_trt == outputs_hf, outputs_trt_ko == outputs_hf_ko],
    ["TensorRT", "FP16", ms(encoder_trt_time_fp16), ms(decoder_trt_time_fp16), ms(full_trt_time_fp16), outputs_trt_fp16 == outputs_hf, outputs_trt_fp16_ko == outputs_hf_ko],
]
print(f"Device: {torch.cuda.get_device_name()} | variant: {Whisper_VARIANT}")
print(tabulate(data, headers="firstrow", tablefmt="github"))

Device: NVIDIA A100-SXM4-80GB | variant: openai/whisper-large-v2
| Framework               | Precision   | Encoder p50 (ms)   | Decoder p50 (ms)   |   Full E2E p50 (ms) | EN == HF   | KO == HF   |
|-------------------------|-------------|--------------------|--------------------|---------------------|------------|------------|
| HuggingFace (w/o cache) | FP32        | -                  | -                  |             1121.82 | -          | -          |
| HuggingFace (w/ cache)  | FP32        | -                  | -                  |              693.1  | -          | -          |
| HuggingFace (w/o cache) | FP16        | -                  | -                  |              775.88 | -          | -          |
| HuggingFace (w/ cache)  | FP16        | -                  | -                  |              560    | -          | -          |
| PyTorch                 | FP32        | 175.16             | 26.43              |              706.55 | True       | -          |
| PyTorch  

In [15]:
# 엔진 컨텍스트 해제
for r in [whisper_trt_encoder, whisper_trt_decoder, whisper_trt_encoder_fp16, whisper_trt_decoder_fp16]:
    r.release()